In [1]:
# ============================================================
# TRUSTSYN TRUST LAYER v2
# Novelty Score
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = Path("/Users/konuri/stacking")

INPUT = (
    BASE /
    "TrustSyn_TRUST_LAYER" /
    "TrustSyn_ensemble_uncertainty.csv"
)

OUTPUT = (
    BASE /
    "TrustSyn_TRUST_LAYER" /
    "TrustSyn_novelty_scores.csv"
)


# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

df = pd.read_csv(INPUT)

print("Loaded:")
print(df.shape)


# ------------------------------------------------------------
# DRUG NOVELTY
# ------------------------------------------------------------

if "Tanimoto_similarity" not in df.columns:
    raise Exception(
        "Tanimoto_similarity missing"
    )


# similarity close to 1 = familiar
# similarity close to 0 = novel

df["drug_novelty"] = (
    1 - df["Tanimoto_similarity"]
)


# ------------------------------------------------------------
# CELL LINE NOVELTY
# ------------------------------------------------------------

pc_cols = [
    f"CellMiner_PC{i}"
    for i in range(1,51)
]


missing = [
    c for c in pc_cols
    if c not in df.columns
]


if missing:
    raise Exception(
        f"Missing CellMiner PCs: {missing}"
    )


# CellMiner PCA matrix

cell_matrix = (
    df[pc_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "Unique cell profiles:",
    cell_matrix.shape
)


# scale PCA space

scaler = StandardScaler()

scaled_cells = scaler.fit_transform(
    cell_matrix
)


# pairwise distance

distance_matrix = euclidean_distances(
    scaled_cells
)


# nearest neighbour distance
# ignore self-distance (0)

distance_matrix[
    distance_matrix == 0
] = np.nan


nearest_distance = (
    np.nanmin(
        distance_matrix,
        axis=1
    )
)


cell_novelty_table = pd.DataFrame(
    {
        "cell_index": range(
            len(cell_matrix)
        ),
        "cell_novelty":
            nearest_distance
    }
)


cell_matrix2 = cell_matrix.copy()

cell_matrix2["cell_index"] = (
    range(len(cell_matrix2))
)


cell_matrix2 = cell_matrix2.merge(
    cell_novelty_table,
    on="cell_index"
)


# map back

df = df.merge(
    cell_matrix2[
        pc_cols +
        ["cell_novelty"]
    ],
    on=pc_cols,
    how="left"
)


# ------------------------------------------------------------
# NORMALIZE CELL NOVELTY
# ------------------------------------------------------------

df["cell_novelty"] = (
    df["cell_novelty"] -
    df["cell_novelty"].min()
) / (
    df["cell_novelty"].max()
    -
    df["cell_novelty"].min()
)


# ------------------------------------------------------------
# FINAL NOVELTY SCORE
# ------------------------------------------------------------

df["novelty_score"] = (
    0.5 * df["drug_novelty"] +
    0.5 * df["cell_novelty"]
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

df.to_csv(
    OUTPUT,
    index=False
)


print("\nNovelty summary")
print(
    df[
        [
            "drug_novelty",
            "cell_novelty",
            "novelty_score"
        ]
    ].describe()
)


print("\nSAVED:")
print(OUTPUT)

Loaded:
(88753, 79)
Unique cell profiles: (59, 50)

Novelty summary
       drug_novelty  cell_novelty  novelty_score
count    990.000000  88753.000000     990.000000
mean       0.926721      0.822953       0.874882
std        0.045012      0.211793       0.107637
min        0.731343      0.000000       0.439655
25%        0.901961      0.728149       0.829872
50%        0.918699      0.896039       0.911140
75%        0.962264      0.968468       0.944336
max        1.000000      1.000000       0.999183

SAVED:
/Users/konuri/stacking/TrustSyn_TRUST_LAYER/TrustSyn_novelty_scores.csv
